In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, zipfile, subprocess
from pathlib import Path
subprocess.run("pip install -q shapely pandas", shell=True)
import pandas as pd
from shapely.geometry import LineString
from shapely.ops import unary_union

BASE_DIR = Path("/content/drive/MyDrive/3-여름학기/미래학술제")
CK = BASE_DIR / "조민서" / "seg_체크포인트"
maps = {**json.load(open(CK / "crack_map.json")), **json.load(open(CK / "pothole_map.json"))}
test_names = sorted(p.name for p in (BASE_DIR / "최종테스트셋" / "최종_원천").glob("*.jpg"))
files = [f for f in test_names if f in maps]

def shoelace(p):
    x, y = p[0::2], p[1::2]
    return 0.5 * abs(sum(x[i]*y[(i+1)%len(y)] - x[(i+1)%len(x)]*y[i] for i in range(len(x))))

def old_area(segs):          # 주혜 함수와 같은 계산
    t = 0.0
    for s in segs:
        if len(s) == 4: t += abs(s[2]*s[3])
        elif len(s) >= 6: t += shoelace(s)
    return t

def crack_new(segs_all, w):  # 모든 균열을 폭 w인 선으로 보고 겹침 없이 합침
    shapes = []
    for s in segs_all:
        pts = list(zip(s[0::2], s[1::2]))
        if len(pts) >= 2 and LineString(pts).length > 0:
            shapes.append(LineString(pts).buffer(w / 2))
    return unary_union(shapes).area if shapes else 0.0

def grade(r): return "긴급" if r >= 15 else ("보통" if r >= 5 else "일반")

rows = []
with zipfile.ZipFile(BASE_DIR / "라벨데이터" / "TL01.zip") as z:
    for fn in files:
        d = json.load(z.open(maps[fn]))
        cat = {c["id"]: c["name"] for c in d["categories"]}
        W, H = d["images"][0]["width"], d["images"][0]["height"]
        tot = W * H
        c_old = p_area = 0.0
        c_segs = []
        for a in d["annotations"]:
            segs = a.get("segmentation") or []
            name = cat.get(a["category_id"], "")
            if "균열" in name:
                c_old += old_area(segs); c_segs += segs
            elif "홀" in name:
                p_area += old_area(segs)
        row = {"사진": fn, "현재_비율(%)": (c_old + p_area) / tot * 100}
        for w in (8, 12, 16):
            row[f"보정w{w}_비율(%)"] = (crack_new(c_segs, w) + p_area) / tot * 100
        rows.append(row)

df = pd.DataFrame(rows)
df["현재_등급"] = df["현재_비율(%)"].map(grade)
df["보정w12_등급"] = df["보정w12_비율(%)"].map(grade)
out = BASE_DIR / "조민서" / "면적비율_재계산.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")
print("저장:", out)
print("\n등급 분포(%)")
for col in ["현재_등급", "보정w12_등급"]:
    print(col, (df[col].value_counts(normalize=True).round(3) * 100).to_dict())
for w in (8, 16):
    print(f"w{w}:", (df[f"보정w{w}_비율(%)"].map(grade).value_counts(normalize=True).round(3) * 100).to_dict())
print("\n등급이 바뀐 사진:", int((df["현재_등급"] != df["보정w12_등급"]).sum()), "/", len(df))

Mounted at /content/drive
저장: /content/drive/MyDrive/3-여름학기/미래학술제/조민서/면적비율_재계산.csv

등급 분포(%)
현재_등급 {'일반': 79.3, '보통': 13.0, '긴급': 7.7}
보정w12_등급 {'일반': 94.3, '보통': 5.7}
w8: {'일반': 94.3, '보통': 5.7}
w16: {'일반': 93.7, '보통': 6.3}

등급이 바뀐 사진: 47 / 300


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import openpyxl, pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/3-여름학기/미래학술제")
CSV  = BASE / "조민서" / "면적비율_재계산.csv"
SRC  = BASE / "조민서" / "위험요소_채점_기록시트_결과입력완료.xlsx"   # 원본 (건드리지 않음)
OUT  = BASE / "조민서" / "위험요소_채점기록시트_면적수정본.xlsx"     # 새로 저장될 파일
WIDTH_COL = "보정w12_비율(%)"                                   # 균열 폭 12px 기준

new = pd.read_csv(CSV).rename(columns={"사진": "파일명"}).set_index("파일명")[WIDTH_COL]

wb = openpyxl.load_workbook(SRC)

# 1) '심각도 신뢰성' 시트: E열(실제 파손 면적 비율) 교체
ws = wb["심각도 신뢰성"]
n1 = 0
for r in range(3, ws.max_row + 1):
    fn = ws.cell(r, 2).value
    if fn:
        ws.cell(r, 5).value = float(new[fn]); n1 += 1

# 2) '파손비율' 시트: 균열면적·총파손면적·파손비율·정답유형 교체
pb = wb["파손비율"]
n2 = chg = 0
for r in range(2, pb.max_row + 1):
    fn = pb.cell(r, 1).value
    if not fn: continue
    tot = float(pb.cell(r, 3).value)       # 이미지 전체 면적
    pot = float(pb.cell(r, 5).value)       # 포트홀 면적(그대로)
    total = float(new[fn]) / 100 * tot
    crack = max(total - pot, 0.0)
    typ = "균열" if crack >= pot else "포트홀"
    chg += (typ != pb.cell(r, 2).value)
    pb.cell(r, 2).value = typ
    pb.cell(r, 4).value = crack
    pb.cell(r, 6).value = total
    pb.cell(r, 7).value = float(new[fn])
    n2 += 1

wb.save(OUT)
print("심각도 신뢰성 갱신:", n1, "줄 / 파손비율 갱신:", n2, "장 / 정답유형 바뀐 수:", chg)
print("저장:", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
심각도 신뢰성 갱신: 600 줄 / 파손비율 갱신: 300 장 / 정답유형 바뀐 수: 19
저장: /content/drive/MyDrive/3-여름학기/미래학술제/조민서/위험요소_채점기록시트_면적수정본.xlsx


In [1]:
import openpyxl
from google.colab import drive
drive.mount("/content/drive")

SRC = "/content/drive/MyDrive/3-여름학기/미래학술제/조민서/위험요소_채점기록시트_면적수정본.xlsx"   # 지금 쓰는 파일명으로
OUT = "/content/drive/MyDrive/3-여름학기/미래학술제/조민서/위험요소_채점기록시트_면적수정본_v2.xlsx"

wb = openpyxl.load_workbook(SRC)

# 새 정답표 (파손비율 시트: 파일명 → 면적 기준 유형)
key = {r[0]: r[1] for r in wb["파손비율"].iter_rows(min_row=2, max_col=2, values_only=True) if r[0]}

ws = wb["AI 채점용"]
changed = 0
for r in range(3, 603):                      # 3~302 HCX, 303~602 Gemini
    new = key[ws.cell(r, 2).value]
    if ws.cell(r, 3).value != new:
        changed += 1
    ws.cell(r, 3).value = new                                        # 정답 유형
    ws.cell(r, 6).value = "O" if ws.cell(r, 4).value == new else "X"  # O/X

print("정답 바뀐 행:", changed)
for name, a, b in [("HCX", 3, 302), ("Gemini", 303, 602)]:
    ox = [ws.cell(r, 6).value for r in range(a, b + 1)]
    print(name, "O", ox.count("O"), "/", len(ox), round(ox.count("O")/len(ox)*100, 1), "%")

wb.save(OUT)
print("저장 완료:", OUT)

Mounted at /content/drive
정답 바뀐 행: 38
HCX O 221 / 300 73.7 %
Gemini O 225 / 300 75.0 %
저장 완료: /content/drive/MyDrive/3-여름학기/미래학술제/조민서/위험요소_채점기록시트_면적수정본_v2.xlsx
